In [ ]:
from figure_paths import DATA, RESULTS, WORK, FIGURES, setting
from pathlib import Path
from contextlib import ExitStack
import json
import os
import tempfile
import shutil
import warnings
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from matplotlib.ticker import MaxNLocator

RESULT_DIR = RESULTS
OUTPUT_DIR = RESULTS / 'task3_typhoon'
FILE_OVERRIDES = {}
LEAD_HOUR = 6
CASE_LEVELS = (850,)  # 默认输出一张10子图；改成(850,925,1000)可逐层输出3张
CASE_NAME = 'Lekima (2019)'
LANDFALL_BJT = '2019-08-10 01:45:00+08:00'
SOURCE_TIMEZONE = 'UTC'
OFFSETS_HOUR = (-24, -12, 0, 12, 24)
MAX_EVENT_MATCH_MINUTES = 30
BOUNDARY_PATH = None  # 本地长江流域 shp/geojson，可同时据其裁剪；不在线下载底图
BASIN_MASK_PATH = None  # 可选：二维NC掩膜，变量名在下方配置；1/True为流域内
BASIN_MASK_VARIABLE = 'basin_mask'
CLIP_TO_BOUNDARY = True
FIG_DPI = 800
FONT_SCALE = 1.0
SHOW_FIGURES = True
SAVE_TIFF = True
SAVE_PDF = True
QUIVER_TARGET_COLUMNS = 20
QUIVER_KEY_MS = None  # None: 根据该图最大风速取易读整数，所有子图统一
NETCDF_TEMP_DIR = str(WORK)
WORK.mkdir(parents=True, exist_ok=True)
MAX_EXTRACT_GIB = 2.0  # 提取49小时、所选层及四个字段的保守内存上限
TIME_ROUND_TOL_NS = 1000
LEVELS = np.asarray(CASE_LEVELS, dtype=float)
ALIASES = {
    'time': ('time', 'valid_time', 'times', 'date', 'datetime'),
    'level': ('level', 'levels', 'pressure_level', 'isobaricInhPa', 'isobaricInPa', 'plev'),
    'latitude': ('latitude', 'lat'), 'longitude': ('longitude', 'lon'),
}
SOURCE_URLS = [
    'https://zj.weather.com.cn/dsxw/08/3226902.shtml',
    'https://qikan.camscma.cn/article/doi/10.11898/1001-7313.20200501',
]
assert FIG_DPI > 600, '正式输出需高于600 dpi。'
assert SOURCE_TIMEZONE == 'UTC', '请先明确源NC的时区；不自动猜测UTC/北京时间。'
assert len(set(CASE_LEVELS)) == len(CASE_LEVELS) and set(CASE_LEVELS) <= {850,925,1000}
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
def rounded_hour_times(values, source):
    raw = np.asarray(values)
    if not np.issubdtype(raw.dtype, np.datetime64):
        raise TypeError(f'{source}: time须可解码为datetime64，不能将非标准日历静默转为标准日历。')
    raw = raw.astype('datetime64[ns]')
    if np.isnat(raw).any() or raw.size == 0:
        raise ValueError(f'{source}: 空时间轴或NaT。')
    ns = raw.astype(np.int64)
    if np.any(np.diff(ns) <= 0):
        raise ValueError(f'{source}: 原时间轴重复或倒序。')
    hour = 3_600_000_000_000
    quotient, remainder = np.divmod(ns, hour)
    rounded_ns = (quotient + (remainder >= hour // 2)) * hour
    delta = rounded_ns - ns
    if np.max(np.abs(delta)) > TIME_ROUND_TOL_NS:
        raise ValueError(f'{source}: 非整点偏差{np.max(np.abs(delta))} ns超过允许的编码噪声。')
    if np.any(np.diff(rounded_ns) <= 0):
        raise ValueError(f'{source}: 整点编码修正后出现重复或倒序。')
    changes = [dict(index=int(i), original=str(raw[i]), rounded=str(np.datetime64(int(rounded_ns[i]), 'ns')),
                    correction_ns=int(delta[i])) for i in np.flatnonzero(delta)]
    return rounded_ns.astype('datetime64[ns]'), changes


def canonical_field(ds, variable, source):
    if variable in ds.data_vars:
        da = ds[variable]
    else:
        candidates = []
        for name, value in ds.data_vars.items():
            if not np.issubdtype(value.dtype, np.number):
                continue
            if all(any(alias in value.dims for alias in aliases) for aliases in ALIASES.values()):
                candidates.append(name)
        if len(candidates) != 1:
            raise ValueError(f'{source}: 找不到{variable}且无法唯一确定四维风场，候选={candidates}。')
        da = ds[candidates[0]]
    rename = {}
    for standard, aliases in ALIASES.items():
        found = [a for a in aliases if a in da.dims]
        if len(found) != 1:
            raise ValueError(f'{source}: {standard}维度不存在或有歧义：{found}。')
        rename[found[0]] = standard
    da = da.rename({a: b for a, b in rename.items() if a != b})
    for dim in tuple(da.dims):
        if dim not in ALIASES:
            if da.sizes[dim] != 1:
                raise ValueError(f'{source}: 不能自动丢弃额外非单例维度{dim}。')
            da = da.isel({dim: 0}, drop=True)
    for dim in ALIASES:
        if dim not in da.coords or da[dim].ndim != 1:
            raise ValueError(f'{source}: {dim}必须有一维坐标。')
    lev = np.asarray(da.level.values, dtype=np.float64)
    units = str(da.level.attrs.get('units', '')).strip().lower().replace(' ', '')
    if units in ('pa', 'pascal', 'pascals') or (not units and np.all(lev > 2000)):
        lev = lev / 100.0
    elif units not in ('', 'hpa', 'hectopascal', 'hectopascals', 'mb', 'millibar', 'millibars'):
        raise ValueError(f'{source}: 未识别的气压层单位{units!r}。')
    if len(np.unique(lev)) != len(lev):
        raise ValueError(f'{source}: 气压层坐标重复。')
    indices = []
    for wanted in LEVELS:
        hit = np.flatnonzero(np.isclose(lev, wanted, rtol=0, atol=1e-8))
        if len(hit) != 1:
            raise ValueError(f'{source}: 缺少或重复气压层{wanted} hPa。')
        indices.append(int(hit[0]))
    da = da.assign_coords(level=lev).isel(level=indices)
    for dim in ('latitude', 'longitude'):
        c = np.asarray(da[dim].values, dtype=np.float64)
        if not np.isfinite(c).all() or len(np.unique(c)) != len(c):
            raise ValueError(f'{source}: {dim}坐标含非有限值或重复。')
        da = da.isel({dim: np.argsort(c)}).assign_coords({dim: np.sort(c)})
    da = da.transpose('time', 'level', 'latitude', 'longitude')
    return da


def source_path(kind, component):
    return Path(FILE_OVERRIDES.get((kind, component), RESULT_DIR /
        f'da_transunet_{kind}_{component}_{LEAD_HOUR}_24year_changjiang_residual.nc'))


def source_stat(path):
    s = Path(path).stat()
    return {'path': str(Path(path).resolve()), 'size': s.st_size, 'mtime_ns': s.st_mtime_ns}


def event_times():
    event = pd.Timestamp(LANDFALL_BJT)
    if event.tzinfo is None:
        raise ValueError('LANDFALL_BJT须明确包含+08:00时区。')
    event = event.tz_convert('UTC')
    anchor = event.round('h')
    shift = (anchor-event).total_seconds()/60
    if abs(shift) > MAX_EVENT_MATCH_MINUTES:
        raise ValueError('登陆与最近整点相差过大。')
    offsets = np.asarray(OFFSETS_HOUR, dtype=int)
    targets = anchor.tz_localize(None).to_datetime64() + offsets.astype('timedelta64[h]')
    all_times = pd.date_range(anchor-pd.Timedelta(hours=24), anchor+pd.Timedelta(hours=24), freq='h')
    return event, anchor, shift, targets.astype('datetime64[ns]'), all_times.tz_localize(None).to_numpy()


def load_basin(latitude, longitude):
    mask = np.ones((len(latitude), len(longitude)), dtype=bool)
    geometry = None
    info = {'boundary': None, 'mask': None, 'domain': 'source grid extent, no explicit basin polygon'}
    if BASIN_MASK_PATH is not None:
        with xr.open_dataset(BASIN_MASK_PATH) as d:
            a = d[BASIN_MASK_VARIABLE]
            rename = {}
            for dim in ('latitude','longitude'):
                hits = [v for v in ALIASES[dim] if v in a.dims]
                if len(hits) != 1:
                    raise ValueError('流域掩膜需要唯一latitude/longitude维度。')
                if hits[0] != dim: rename[hits[0]] = dim
            a = a.rename(rename).transpose('latitude','longitude').sortby('latitude').sortby('longitude')
            if not (np.array_equal(a.latitude,latitude) and np.array_equal(a.longitude,longitude)):
                raise ValueError('流域掩膜和预测网格不一致；不自动插值。')
            values = np.asarray(a.values)
            if not np.isin(values[np.isfinite(values)], [0,1]).all():
                raise ValueError('流域掩膜有限值必须为0/1。')
            mask &= np.isfinite(values) & (values == 1)
        info['mask'] = source_stat(BASIN_MASK_PATH)
        info['domain'] = 'explicit basin mask'
    if BOUNDARY_PATH is not None:
        import geopandas as gpd
        import shapely
        geometry = gpd.read_file(BOUNDARY_PATH)
        if geometry.crs is None:
            raise ValueError('边界文件缺少CRS，不能猜测坐标系。')
        geometry = geometry.to_crs('EPSG:4326')
        if geometry.empty or geometry.geometry.is_empty.all():
            raise ValueError('边界文件为空。')
        if CLIP_TO_BOUNDARY:
            union = geometry.geometry.union_all() if hasattr(geometry.geometry,'union_all') else geometry.geometry.unary_union
            xx, yy = np.meshgrid(longitude, latitude)
            if hasattr(shapely, 'intersects_xy'):
                within = shapely.intersects_xy(union, xx, yy)
            else:
                from shapely.geometry import Point
                within = np.array([union.covers(Point(x,y)) for x,y in zip(xx.ravel(),yy.ravel())]).reshape(xx.shape)
            mask &= within
            info['domain'] = 'explicit basin boundary clipped at grid centres'
        info['boundary'] = source_stat(BOUNDARY_PATH)
    if not mask.any():
        raise ValueError('流域掩膜为空。')
    if BASIN_MASK_PATH is None and (BOUNDARY_PATH is None or not CLIP_TO_BOUNDARY):
        warnings.warn('没有指定流域裁剪文件：保留源NC的有效网格，不把矩形网格外框声称为长江流域边界。若源NC尚未裁剪，请配置BOUNDARY_PATH或BASIN_MASK_PATH。')
    return mask, geometry, info


def read_case():
    event, anchor, shift, targets, window = event_times()
    records, fields, lat, lon = [], {}, None, None
    for kind in ('testy','predicty'):
        for component in ('u','v'):
            path = source_path(kind, component)
            if not path.is_file(): raise FileNotFoundError(path)
            before = source_stat(path)
            with xr.open_dataset(path, cache=False) as ds:
                a = canonical_field(ds, f'{kind}_{component}', path)
                times, corrections = rounded_hour_times(a.time.values, path)
                if lat is None:
                    lat,lon = a.latitude.values.copy(),a.longitude.values.copy()
                    estimated = len(window)*len(LEVELS)*len(lat)*len(lon)*8*12/2**30
                    if estimated > MAX_EXTRACT_GIB:
                        raise MemoryError(f'提取和绘图保守估计{estimated:.2f}GiB，超过MAX_EXTRACT_GIB。可一次仅处理1层。')
                elif not (np.array_equal(lat,a.latitude) and np.array_equal(lon,a.longitude)):
                    raise ValueError('u/v/testy/predicty网格不一致。')
                idx = np.searchsorted(times,window)
                if np.any(idx>=len(times)) or not np.array_equal(times[np.minimum(idx,len(times)-1)],window):
                    missing = window[~np.isin(window,times)]
                    raise ValueError(f'{path}: 缺少事件窗口时次{missing}；不补造或自动换事件。')
                fields[f'{kind}_{component}'] = a.isel(time=idx).assign_coords(time=window).load()
            if before != source_stat(path): raise RuntimeError('读取期间源文件发生变化，请重新运行。')
            records.append({**before, 'kind':kind, 'component':component, 'encoding_time_corrections':corrections})
    mask, geometry, domain_info = load_basin(lat,lon)
    ds = xr.Dataset(fields)
    valid = np.isfinite(ds.testy_u) & np.isfinite(ds.testy_v) & np.isfinite(ds.predicty_u) & np.isfinite(ds.predicty_v)
    valid &= xr.DataArray(mask, dims=('latitude','longitude'),coords={'latitude':lat,'longitude':lon})
    if not valid.any().item(): raise ValueError('事件窗口没有共同有效的u/v样本。')
    for name in fields:
        ds[name] = ds[name].where(valid)
        ds[name].attrs['units'] = 'm s-1'
    ds['common_valid_mask'] = valid.astype('uint8')
    for kind in ('testy','predicty'):
        ds[f'{kind}_speed'] = np.hypot(ds[f'{kind}_u'], ds[f'{kind}_v'])
        ds[f'{kind}_speed'].attrs['units'] = 'm s-1'
    ds = ds.assign_coords(forecast_initialization_time=('time',window-np.timedelta64(LEAD_HOUR,'h')))
    ds.attrs.update(event_name=CASE_NAME, lead_hour=LEAD_HOUR, landfall_utc=event.isoformat(),
                    anchor_utc=anchor.isoformat(), anchor_offset_minutes=shift,
                    time_convention='UTC target valid time; initialization=valid time minus lead',
                    selection='historically documented event; not selected using prediction error',
                    domain=domain_info['domain'])
    rows=[]
    for offset,target in zip(OFFSETS_HOUR,targets):
        stamp=pd.Timestamp(target,tz='UTC')
        rows.append({'nominal_offset_hour':offset,'target_utc':stamp.isoformat(),
                     'target_bjt':stamp.tz_convert('Asia/Shanghai').isoformat(),
                     'actual_hours_since_landfall':(stamp-event).total_seconds()/3600,
                     'initialization_utc':(stamp-pd.Timedelta(hours=LEAD_HOUR)).isoformat()})
    pd.DataFrame(rows).to_csv(OUTPUT_DIR/'selected_times.csv',index=False,encoding='utf-8-sig')
    audit={'event':CASE_NAME,'landfall_bjt':str(event.tz_convert('Asia/Shanghai')),
           'landfall_utc':str(event),'nearest_hour_utc':str(anchor),'rounding_minutes':shift,
           'window_hourly_count':len(window),'sources':SOURCE_URLS,'files':records,
           'domain':domain_info,'selected_times':rows,
           'limits':'The case is not a basin-wide percentile-extreme classification; evaluate that separately.'}
    (OUTPUT_DIR/'case_audit.json').write_text(json.dumps(audit,ensure_ascii=False,indent=2),encoding='utf-8')
    return ds, geometry, targets, audit


def save_netcdf(ds, filename):
    with tempfile.TemporaryDirectory(prefix='typhoon_case_',dir=NETCDF_TEMP_DIR) as tmp:
        stage=Path(tmp)/'case.nc'
        ds.to_netcdf(stage,engine='netcdf4',encoding={k:{'zlib':True,'complevel':4} for k in ds.data_vars})
        with xr.open_dataset(stage) as check:
            for key in ds.data_vars:
                if not np.array_equal(ds[key].values,check[key].values,equal_nan=True):
                    raise IOError(f'写出回读不一致：{key}')
        target=OUTPUT_DIR/filename
        temp=target.with_suffix('.tmp.nc')
        shutil.copyfile(stage,temp); os.replace(temp,target)


In [ ]:
case, boundary, selected_times, audit = read_case()
save_netcdf(case, 'Lekima_48h_6h_forecasts.nc')
